<a href="https://colab.research.google.com/github/ANEESAAYAZ/internship_feature_engineering/blob/main/internship_FeatureEngineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 60)
SNAPSHOT=pd.Timestamp("2024-05-01")

In [2]:
df=pd.read_csv("properties_clean.csv",parse_dates=["listed_date"])
print(f"loaded dataset has {df.shape[0]} rows and {df.shape[1]} columns")

loaded dataset has 3000 rows and 42 columns


In [3]:
df["sold_fast"]=np.where(df["listing_status"]=="Active",np.nan,(df["days_on_market"]<=30).astype(float))


In [4]:
df["sold_fast"][:10]

,sold_fast
0,0.0
1,0.0
2,0.0
3,1.0
4,1.0
5,1.0
6,0.0
7,0.0
8,0.0
9,0.0


In [5]:
df["sold_fast"][:10].tolist()

[0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0]

In [6]:
print(f"\n Target 1 - price_aed : Regression {df['price_aed'].notna().sum()} rows usable")
print(f"\n Target 2 - sold_fast : Classification {df['sold_fast'].notna().sum()} rpws usable"
      f" ({df["sold_fast"].mean():.1%}) positive")


 Target 1 - price_aed : Regression 3000 rows usable

 Target 2 - sold_fast : Classification 2563 rpws usable (38.7%) positive


**Combine**

In [7]:
df["price_per_sqft"]=df["price_aed"]/df["built_up_area_sqft"]
df["rental_yield_pct"]=100*df["annual_rent_aed"]/df["price_aed"]
df["bath_per_bed"]=df["bathrooms"]/df["bedrooms"].replace(0,1)
df["service_charge_annual"]=df["service_charge_aed_sqft"]*df["built_up_area_sqft"]
df["sqft_per_bed"]=df["built_up_area_sqft"]/df["bedrooms"].replace(0,1)
df["plot_ratio"]=np.where(df["has_plot_area_sqft"]==1,df["plot_area_sqft"]/df["built_up_area_sqft"],np.nan)

In [8]:
print("\n--------price per sqft by emirate (the number every agent quote)--------")
print(df.groupby("emirate")["price_per_sqft"].median().round(0).sort_values())

print("\n----------rental yield plausible-----------")
print(df["rental_yield_pct"].describe().round(2))

print("\nUAE market reality:5%-9% is normal,above 11% is bad data")


--------price per sqft by emirate (the number every agent quote)--------
emirate
Ajman              390.0
Sharjah            923.0
Abu Dhabi         1375.0
Ras Al Khaimah    1426.0
Dubai             1740.0
Name: price_per_sqft, dtype: float64

----------rental yield plausible-----------
count    1554.00
mean        7.06
std         1.99
min         0.45
25%         5.79
50%         6.74
75%         8.02
max        19.61
Name: rental_yield_pct, dtype: float64

UAE market reality:5%-9% is normal,above 11% is bad data


**Decode**

In [9]:
df["listing_month"]=df["listed_date"].dt.month
df["listing_quarter"]=df["listed_date"].dt.quarter
df["listing_year"]=df["listed_date"].dt.year
df["listing_dayofweek"]=df["listed_date"].dt.dayofweek
df["days_since_listed"]=(SNAPSHOT - df["listed_date"]).dt.days
df["is_summer_listing"]=df["listing_month"].isin([6,7,8]).astype(int)
df["is_winter_listing"]=df["listing_month"].isin([12,1,2]).astype(float)
df["month_sin"]=np.sin(2*np.pi*df["listing_month"]/12)
df["month_cos"]=np.cos(2*np.pi*df["listing_month"]/12)

In [10]:

print("\n------listing per month collection------")
print(df["listed_date"].dt.to_period("M").value_counts().sort_index().tail(10).to_string())


------listing per month collection------
listed_date
2023-07      19
2023-08      32
2023-09      41
2023-10      83
2023-11     139
2023-12     142
2024-01     261
2024-02     478
2024-03     727
2024-04    1040
Freq: M


In [11]:
settled=df[df["sold_fast"].notna()]
print("\n----Sold_fast rate by listing month(watch the counts-----)")
print(settled.groupby("listing_month")["sold_fast"].agg(["count","mean"]).round(3).to_string())



----Sold_fast rate by listing month(watch the counts-----)
               count   mean
listing_month              
1                216  0.343
2                410  0.437
3                622  0.383
4                894  0.385
5                  7  0.000
6                  3  0.000
7                 19  0.263
8                 34  0.353
9                 38  0.368
10                76  0.421
11               125  0.368
12               119  0.395


In [12]:
print(f"\nSummer listings : {settled.loc[settled["is_summer_listing"]==1,"sold_fast"].mean():.1%}"
      f"sold fast   (n={int((settled["is_summer_listing"]==1).sum())})")
print(f"\nNon Summer listings : {settled.loc[settled["is_summer_listing"]==0,"sold_fast"].mean():.1%}"
      f"sold fast   (n={int((settled["is_summer_listing"]==0).sum())})")



Summer listings : 30.4%sold fast   (n=56)

Non Summer listings : 38.9%sold fast   (n=2507)


In [13]:
df["property_age_year"]=SNAPSHOT.year-df["year_of_completion"]
df["is_new_build"]=(df["property_age_year"]<=2).astype(int)
df["is_off_plan"]=(df["completion_status"]=="off_plan").astype(int)
df["age_is_known"]=df["property_age_year"].notna().astype(int)
print(df["property_age_year"].describe().round(1))

count    1052.0
mean        7.5
std         5.6
min         0.0
25%         2.0
50%         5.5
75%        13.0
max        19.0
Name: property_age_year, dtype: float64


In [14]:
print(f"unknown age: {df["property_age_year"].isna().sum()} of {len(df)}")

unknown age: 1948 of 3000


In [15]:
print("\n-------is the age missing at random-------")
print(df.assign(unknown=df["property_age_year"].isna())
      .groupby("completion_status")["unknown"]
      .agg(["count","mean"]).round(3).to_string())
df["property_age_year"]=df["property_age_year"].clip(lower=0)


-------is the age missing at random-------
                   count   mean
completion_status              
Off-Plan            1196  0.995
Ready               1804  0.420


**Reshape**

In [16]:
for col in ["price_aed","built_up_area_sqft","annual_rent_aed","service_charge_annual"]:
  df[f"log_{col}"]=np.log1p(df[col])
print("\n---Skewness before and log1p-----\n")
for col in ["price_aed","built_up_area_sqft","annual_rent_aed","service_charge_annual"]:
  print(f"   {col:22s}   {df[col].skew():7.2f}    -> {df[f"log_{col}"].skew():6.2f}")



---Skewness before and log1p-----

   price_aed                   6.76    ->   0.51
   built_up_area_sqft          3.22    ->   0.52
   annual_rent_aed             5.44    ->   0.38
   service_charge_annual       5.83    ->   0.32


In [17]:
# We taking log values to reduce skewness as it may affect the prediction, prediction errors


**Summarise**

In [18]:
from sklearn.model_selection import KFold

def cross_fitted_target_encode(frame,key_col,target_col,n_splits=5,seed=42):
  out=pd.Series(np.nan,index=frame.index,dtype=float)
  usable=frame[frame[target_col].notna()]
  global_mean=usable[target_col].mean()

  kf=KFold(n_splits=n_splits,shuffle=True,random_state=seed)
  for fit_idx,apply_idx in kf.split(usable):
    fit_rows=usable.iloc[fit_idx]
    apply_rows=usable.iloc[apply_idx]
    means=fit_rows.groupby(key_col)[target_col].mean()
    out.loc[apply_rows.index]=apply_rows[key_col].map(means).fillna(global_mean).values
  return out.fillna(global_mean)
df["area_price_encoded"]=cross_fitted_target_encode(df,"area","log_price_aed")
naive=df.groupby("area")["log_price_aed"].transform("mean")

print("\n-----Correlation with log_price_aed----")
print("==========================================")
print(f"naive target encoding        :{naive.corr(df["log_price_aed"]):.4f}")
print(f"cross-fitted target encoding :{df["area_price_encoded"].corr(df["log_price_aed"]):.4f}")
print(f"\n({df["area"].nunique()} communities compressed into 1 column either way)")


print("\n----community size,and why rare ones are risky----")
print("===================================================")
sizes=df["area"].value_counts()
print(f"    Communities with >=50 listings   :{(sizes>=50).sum()}")
print(f"    Communities with >=10 listings   :{(sizes>=10).sum()}")
print(f"    Communities with >=5 listings    :{(sizes>=5).sum()}")



-----Correlation with log_price_aed----
naive target encoding        :0.7421
cross-fitted target encoding :0.7191

(100 communities compressed into 1 column either way)

----community size,and why rare ones are risky----
    Communities with >=50 listings   :13
    Communities with >=10 listings   :99
    Communities with >=5 listings    :100


***Geographical Feature Engineering***

In [19]:
LANDMARKS = {
    "burj_khalifa": (25.19720, 55.27440),
    "dxb_airport": (25.25280, 55.36440),
    "dubai_marina": (25.08000, 55.14000),
    "abu_dhabi_corniche": (24.46390, 54.32220),
}


In [20]:
def haversine_km(lat1, lon1, lat2, lon2):
  r = 6371.0088
  p1, p2 = np.radians(lat1), np.radians(lat2)
  dphi = np.radians(lat2 - lat1)
  dlmb = np.radians(lon2 - lon1)
  a = (
        np.sin(dphi / 2) ** 2
        + np.cos(p1) * np.cos(p2)
        * np.sin(dlmb / 2) ** 2
  )
  return 2 * r * np.arcsin(np.sqrt(a))



In [21]:
for name, (lat, lon) in LANDMARKS.items():

    df[f"km_to_{name}"] = haversine_km(
        df["latitude"],
        df["longitude"],
        lat,
        lon
    ).round(2)

In [22]:
df["near_metro"] = (
    (df["distance_to_metro_km"] >= 0) &
    (df["distance_to_metro_km"] <= 1.0)
).astype(int)


In [24]:
dist_cols = [c for c in df.columns if c.startswith("km_to_")] + \
            ["distance_to_downtown_km", "distance_to_metro_km"]

print("\n-------Correlation of each distance with log price---------- ")
print("==============================================================")
df[dist_cols + ["log_price_aed"]].corr()
print(
    df[dist_cols + ["log_price_aed"]]
    .corr()["log_price_aed"]
    .drop("log_price_aed")
    .round(3)
    .to_string()
)
print(
    f"\nwithin 1 km of a metro station: "
    f"{df['near_metro'].sum()} listings"
)


-------Correlation of each distance with log price---------- 
km_to_burj_khalifa         -0.008
km_to_dxb_airport           0.035
km_to_dubai_marina         -0.048
km_to_abu_dhabi_corniche   -0.108
distance_to_downtown_km    -0.046
distance_to_metro_km        0.006

within 1 km of a metro station: 690 listings


In [26]:
print("\n"+ "=" *72)
print("WHICH FEATURES ARE LEGAL FOR WHICH TARGET?")
print("="*72)
print("""
The test,applied to every feature:
    "would I know this value at the moment I need prediction?"
Predicting PRICE means the property is not yet priced.
Predicting SOLD_FAST heppens the day the listing goes live,price already set.

Feature                   Price_aed                 sold_fast
----------------------------------------------------------------
built_up_area_sqft        legal                       legal
property_agr_years        legal                       legal
is_summer_listing         legal                       legal
price_per_sqft            LEAKS(needs p)              legal
rental_yield_pct          LEAKS(needs p)              legal
log_price_aed             Is the target               legal - a strong predictor
area_price_encoded        built from target           legal
price_reduced_count       LEAKS                       LEAKS - only known afterward
days_on_market            legal                       Is the target
""")


PRICE_FORBIDDEN=["price_per_sqft","rental_yield_pct","log_price_aed","price_reduced_count","days_on_market","area_price_encoded","sold_fast","listing_status"]

SOLD_FAST_FORBIDDEN=["price_reduced_count","days_on_market","sold_fast","listing_status"]

print(f"drop when modeling price_aed: {len(PRICE_FORBIDDEN)}  columns")
print(f"\ndrop when modeling sold_fast: {len(SOLD_FAST_FORBIDDEN)} columns")


WHICH FEATURES ARE LEGAL FOR WHICH TARGET?

The test,applied to every feature:
    "would I know this value at the moment I need prediction?"
Predicting PRICE means the property is not yet priced.
Predicting SOLD_FAST heppens the day the listing goes live,price already set.

Feature                   Price_aed                 sold_fast
----------------------------------------------------------------
built_up_area_sqft        legal                       legal
property_agr_years        legal                       legal
is_summer_listing         legal                       legal
price_per_sqft            LEAKS(needs p)              legal
rental_yield_pct          LEAKS(needs p)              legal
log_price_aed             Is the target               legal - a strong predictor
area_price_encoded        built from target           legal
price_reduced_count       LEAKS                       LEAKS - only known afterward
days_on_market            legal                       Is the target

dro

In [27]:
df=df.drop(columns=["price_reduced_count"],errors="ignore")
print("\ndropped price_reduced_count - unknowable at prediction time for BOTH targets")

df.to_csv("properties_features.csv",index=False)
print(f"saved data/properties_feature.csv {df.shape}")


dropped price_reduced_count - unknowable at prediction time for BOTH targets
saved data/properties_feature.csv (3000, 71)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Select properties with known prices
model_df = df[df["price_aed"].notna()].copy()

# Transform target using logarithm
y = np.log1p(model_df["price_aed"])


# Original numerical features
BASE_NUM = [
    "built_up_area_sqft", "bedrooms", "bathrooms",
    "floor_number", "distance_to_downtown_km",
    "service_charge_aed_sqft"
]

# Original + engineered numerical features
ENG_NUM = BASE_NUM + [
    "property_age_years", "bath_per_bed",
    "sqft_per_bed", "log_built_up_area_sqft",
    "plot_ratio", "near_metro", "is_offplan",
    "is_summer_listing", "month_sin", "month_cos",
    "km_to_burj_khalifa"
]

# Categorical features
CAT = [
    "emirate", "area", "property_type",
    "furnishing", "view"
]


def evaluate(numeric_cols, label):

    # Preprocessing
    prep = ColumnTransformer([
        ("num", Pipeline([
            ("i", SimpleImputer(strategy="median")),
            ("s", StandardScaler())
        ]), numeric_cols),

        ("cat", Pipeline([
            ("i", SimpleImputer(strategy="most_frequent")),
            ("e", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ))
        ]), CAT),
    ])

    # Complete preprocessing + model pipeline
    pipe = Pipeline([
        ("prep", prep),
        ("model", Ridge(alpha=1.0))
    ])

    # Five-fold cross-validation
    s = cross_val_score(
        pipe,
        model_df,
        y,
        cv=5,
        scoring="r2"
    )

    # Display average score and standard deviation
    print(
        f"  {label:34s} "
        f"R2 = {s.mean():.4f} +/- {s.std():.4f}"
    )

    return s.mean()
